# 🌐 Run Virgo-1.0-Angkor for the Virgo AI website — Google Vertex AI (Colab Enterprise)

Runs Virgo's server (chat **12B**, Virgo's Khmer hearing, and **Virgo-1.0-Angkor-Voice**) on a **Google Cloud L4 GPU** with 32 GB of RAM: everything fits on one machine, nothing is squeezed, and it can keep running without a browser tab. It serves your own address **virgo.camksn.com** (Cloudflare Tunnel), so nothing changes on the website.

Cost: about **$0.85–1/hour** on-demand for g2-standard-8 + L4 (≈ $20–24 a day if left on 24/7; Spot ≈ 60–70% less but can be stopped). Prices change: check Google Cloud pricing. **Stop the runtime when you don't need Virgo** to save money.

**First time only (Google Cloud)**
1. Create a project at [console.cloud.google.com](https://console.cloud.google.com) and turn on billing (new accounts get **$300 free credit**).
2. **IAM & Admin → Quotas**: request **1 × NVIDIA L4 GPU** (`NVIDIA_L4_GPUS`) in a region such as `us-central1`. New projects start with 0.
3. Turn on the **Vertex AI API** and the **Secret Manager API**.
4. **Security → Secret Manager → Create secret** for each secret below (name exactly as written, value = the token). Grant your Colab runtime's service account the **Secret Manager Secret Accessor** role. (Or skip this: the notebook asks for each one when it runs.)
5. **Vertex AI → Colab Enterprise → Runtime templates → New**: machine **g2-standard-8** (1 × L4, 32 GB RAM), disk 200 GB. Tick **Spot** to pay ~60–70% less (it can be stopped; just run again).

**Run**: Colab Enterprise → **Import notebook** from GitHub (`nangmrr752/virgo-models`, this file) → connect to your template's runtime → **Run all**.

**Secrets:** `HF_TOKEN`, `CF_TUNNEL_TOKEN` (your tunnel token) and `VIRGO_API_KEY` (the same password the website uses).

**Keep it running without the tab:** Colab Enterprise → **Executions** → run this notebook on your template (with the secrets in Secret Manager and `ASK_FOR_SECRETS = False`). The last cell keeps the server running until you cancel the execution.

In [ ]:
# ⚙️ Settings
HF_MODEL = ""  # leave empty to use <your Hugging Face name>/Virgo-1.0-Angkor
VIRGO_HOST = "https://virgo.camksn.com"  # your own address (named Cloudflare Tunnel), used when the CF_TUNNEL_TOKEN secret is set
ASK_FOR_SECRETS = True  # False for background Executions (use Secret Manager)

In [ ]:
# 1️⃣ Set up
import os, subprocess, sys, torch
assert torch.cuda.is_available(), "❌ No GPU: connect to a runtime template with an NVIDIA L4 (see the steps above)"
subprocess.run("pkill -f 'uvicorn serve.app' ; pkill -f cloudflared", shell=True)
WORK = "/content" if os.path.isdir("/content") else os.path.expanduser("~")
os.chdir(WORK)
if not os.path.isdir("virgo-models/.git"):
    r = subprocess.run(["git", "clone", "-q", "https://github.com/nangmrr752/virgo-models"])
    assert r.returncode == 0, "❌ Can't download the Virgo code from GitHub"
os.chdir(f"{WORK}/virgo-models")
subprocess.run(["git", "pull", "-q"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-serve.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kokoro"])  # natural English voice; optional (MMS is used if it fails)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])  # conflicts with newer peft
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-cloud-secret-manager"])
def secret(name):
    """A secret: from the environment, else Google Secret Manager (this project), else asked once."""
    if os.environ.get(name):
        return os.environ[name]
    try:
        import google.auth
        from google.cloud import secretmanager
        _, project = google.auth.default()
        value = secretmanager.SecretManagerServiceClient().access_secret_version(
            name=f"projects/{project}/secrets/{name}/versions/latest").payload.data.decode().strip()
        os.environ[name] = value
        return value
    except Exception:
        pass
    if ASK_FOR_SECRETS:
        from getpass import getpass
        value = getpass(f"{name} (leave empty to skip): ").strip()
        if value:
            os.environ[name] = value
        return value or None
    return None

from huggingface_hub import login, HfApi, snapshot_download
token = secret("HF_TOKEN")
assert token, "❌ No HF_TOKEN: add it in Secret Manager (see the steps above) or type it when asked"
login(token=token)
ADAPTER = "chat/out/virgo-1.0-chat-lora"
if not os.path.isdir(ADAPTER):
    me = HfApi().whoami()['name']
    # Virgo-1.0-Angkor-12B (chat/train_kaggle.ipynb) when you've trained it, else the 4B.
    repo = HF_MODEL or next((r for r in (f"{me}/Virgo-Angkor-1.0-12B", f"{me}/Virgo-1.0-Angkor-12B", f"{me}/Virgo-Angkor-1.0-4B", f"{me}/Virgo-1.0-Angkor")
                             if HfApi().file_exists(r, "adapter_config.json")), f"{me}/Virgo-1.0-Angkor")
    print("⬇️ Downloading", repo)
    snapshot_download(repo, local_dir=ADAPTER)
# Virgo-1.0-Bayon (chat/train_tpu_kaggle.ipynb), when you've trained it: served at the same address,
# picked with "model": "virgo-1.0-bayon". It needs its own GPU (Kaggle T4 x2) or a 24 GB+ GPU.
SERVE_BAYON = True
BAYON = "chat/out/virgo-1.0-bayon"
os.environ["VIRGO_CHAT_MODELS"] = f"virgo-1.0-angkor={ADAPTER}"
gpus = torch.cuda.device_count()
has_bayon = False
for bayon_repo in (f"{HfApi().whoami()['name']}/{n}" for n in ("Virgo-Bayon-1.0-27B", "Virgo-1.0-Bayon", "Virgo-Bayon-1.0-12B", "Virgo-1.0-Bayon-12B", "Virgo-Bayon-1.0-4B", "Virgo-1.0-Bayon-4B")):
    try:
        has_bayon = SERVE_BAYON and (HfApi().file_exists(bayon_repo, "config.json") or HfApi().file_exists(bayon_repo, "adapter_config.json"))
    except Exception:
        has_bayon = False
    if has_bayon:
        break
sys.path.append("serve")
from fit import bayon_fits
room = has_bayon and bayon_fits(bayon_repo)
if has_bayon and not room:
    print(f"ℹ️ {bayon_repo} won't fit next to Angkor on these GPUs: serving Angkor only.")
elif has_bayon:
    if not os.path.isdir(BAYON):
        print("⬇️ Downloading", bayon_repo)
        snapshot_download(bayon_repo, local_dir=BAYON)
    os.environ["VIRGO_CHAT_MODELS"] += f",virgo-1.0-bayon={BAYON}"
    print("✅ Also serving Virgo-1.0-Bayon at the same address")
# Virgo's own Khmer hearing (speech/train_hearing_kaggle.ipynb), when you've trained it.
HEARING = "speech/out/virgo-1.0-stt"
if not os.path.isdir(HEARING):
    try:
        hearing_repo = next((f"{HfApi().whoami()['name']}/{n}" for n in ("Virgo-Angkor-1.0-STT", "Virgo-1.0-Angkor-Hearing") if HfApi().repo_exists(f"{HfApi().whoami()['name']}/{n}")), f"{HfApi().whoami()['name']}/Virgo-1.0-Angkor-Hearing")
        if HfApi().file_exists(hearing_repo, "config.json"):
            print("⬇️ Downloading Virgo's Khmer hearing")
            snapshot_download(hearing_repo, local_dir=HEARING)
    except Exception as err:
        print("Virgo's own hearing not used:", err)
print("✅ Ready: Virgo-1.0-Angkor is in", ADAPTER, "| hearing:", "Virgo's own" if os.path.isdir(HEARING) else "Whisper large-v3-turbo")

In [ ]:
# 🗣️ Virgo-1.0-Angkor-Voice: Virgo's voice on VoxCPM2 (Apache 2.0, clear Khmer + English)
# Uses your designed voice (speech/design_voice.ipynb → you/Virgo-1.0-Angkor-Voice) when it exists,
# otherwise a described warm, clear voice. Set USE_VIRGO_VOICE = False to use the older voices.
USE_VIRGO_VOICE = True
import os, subprocess, sys, torch
from huggingface_hub import HfApi
if USE_VIRGO_VOICE:
    env_dir = f"{WORK}/vox-env"
    python = f"{env_dir}/bin/python"
    if not os.path.exists(python):
        print("⏳ Installing VoxCPM2 for Virgo's voice (3–5 minutes, first run only)...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
        subprocess.run(["uv", "venv", "-q", "--python", "3.11", env_dir], check=True)
        r = subprocess.run(["uv", "pip", "install", "-q", "--python", python, "git+https://github.com/OpenBMB/VoxCPM", "soundfile", "huggingface_hub"], capture_output=True, text=True)
        if r.returncode:  # GitHub unreachable: the pip release
            r = subprocess.run(["uv", "pip", "install", "-q", "--python", python, "voxcpm", "soundfile", "huggingface_hub"], capture_output=True, text=True)
        if r.returncode:
            print(r.stderr[-3000:]); raise SystemExit("❌ VoxCPM2 didn't install: send the lines above to Claude.")
    # English: Kokoro-82M (Apache 2.0), a separate fast voice, in the same environment (it needs Python < 3.13).
    if subprocess.run([python, "-c", "import kokoro"], capture_output=True).returncode:
        subprocess.run("apt-get -qq install -y espeak-ng > /dev/null 2>&1", shell=True)
        r = subprocess.run(["uv", "pip", "install", "-q", "--python", python, "kokoro>=0.9"], capture_output=True, text=True)
        if r.returncode:
            print("⚠️ Kokoro (English voice) didn't install, VoxCPM2 will speak English too:", r.stderr[-500:])
    # Download VoxCPM2 now (about 5 GB) so the first answer isn't slow.
    subprocess.run([python, "-c", "from huggingface_hub import snapshot_download; snapshot_download('openbmb/VoxCPM2')"], check=True)
    os.environ["VIRGO_VOX_PYTHON"] = python
    os.environ["VIRGO_VOX_REPO"] = next((f"{HfApi().whoami()['name']}/{n}" for n in ("Virgo-Bakong-2.0-TTS", "Virgo-Bakong-2.0-Voice", "Virgo-Angkor-1.0-TTS") if HfApi().repo_exists(f"{HfApi().whoami()['name']}/{n}")), f"{HfApi().whoami()['name']}/Virgo-1.0-Angkor-Voice")
    if torch.cuda.device_count() > 1:
        os.environ["VIRGO_VOX_DEVICE"] = "cuda:1"  # the second T4: chat and hearing keep the first
    try:
        designed = HfApi().file_exists(os.environ["VIRGO_VOX_REPO"], "voice.wav")
    except Exception:
        designed = False
    print("✅ Virgo speaks with", "your designed voice from " + os.environ["VIRGO_VOX_REPO"] if designed else "the described voice (design yours with speech/design_voice.ipynb)",
          "| GPU:", os.environ.get("VIRGO_VOX_DEVICE", "cuda:0"))
    if torch.cuda.get_device_properties(0).total_memory < 20e9:
        # A small GPU (T4, 16 GB): chat in 4 bits and no torch.compile for the voice, so all fit. An L4 (24 GB) needs neither.
        os.environ["VIRGO_CHAT_4BIT"] = "1"
        os.environ["VIRGO_VOX_OPTIMIZE"] = "0"
        print("ℹ️ One GPU: chat runs in 4 bits so Virgo's voice fits too. Use the 4B Virgo-1.0-Angkor (HF_MODEL in Settings).")

In [ ]:
# 🎙️ Virgo's own Khmer voice (when trained with speech/train_voice_kaggle.ipynb)
import os, subprocess, sys
from huggingface_hub import HfApi
VOICE_REPO = f"{HfApi().whoami()['name']}/Virgo-1.0-Angkor-Voice"
try:
    has_voice = HfApi().file_exists(VOICE_REPO, "model.pth")
except Exception:
    has_voice = False
if has_voice:
    # The voice needs its own Python environment (older libraries than the Virgo server).
    env_dir = f"{WORK}/voice-env"
    python = f"{env_dir}/bin/python"
    if not os.path.exists(python):
        print("⏳ Installing Virgo's Khmer voice (2–4 minutes, first run only)...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
        subprocess.run(["uv", "venv", "-q", "--python", sys.executable, env_dir], check=True)
        subprocess.run(["uv", "pip", "install", "-q", "--python", python, "--index-url", "https://download.pytorch.org/whl/cpu", "torch==2.8.*", "torchaudio==2.8.*"], check=True)
        subprocess.run(["uv", "pip", "install", "-q", "--python", python, "coqui-tts==0.27.2", "soundfile"], check=True)
    os.environ["VIRGO_VOICE_PYTHON"] = python
    os.environ["VIRGO_VOICE_REPO"] = VOICE_REPO
    print("✅ Khmer answers will use Virgo's own voice from", VOICE_REPO)
else:
    print("ℹ️ No trained Virgo voice yet: Khmer uses the MMS voice (non-commercial). Train one with speech/train_voice_kaggle.ipynb.")

In [ ]:
# 2️⃣ Start the Virgo server and open a public address
import os, subprocess, time, re, secrets, json, urllib.request
os.chdir(f"{WORK}/virgo-models")
SAVED_KEY = secret("VIRGO_API_KEY")
KEY = SAVED_KEY or secrets.token_urlsafe(32)  # a saved key stays the same every run
TUNNEL = secret("CF_TUNNEL_TOKEN")           # your own tunnel: the address never changes
env = {**os.environ, "VIRGO_API_KEY": KEY, "VIRGO_PRELOAD": "chat,speech"}
if "virgo-1.0-bayon" not in os.environ.get("VIRGO_CHAT_MODELS", ""):
    env["CUDA_VISIBLE_DEVICES"] = "0"  # one model: keep Virgo on the first GPU
subprocess.Popen(f"uvicorn serve.app:app --host 0.0.0.0 --port 8000 > {WORK}/server.log 2>&1", shell=True, env=env)
print("⏳ Loading Virgo-1.0-Angkor (1–3 minutes)...")
for _ in range(180):
    time.sleep(2)
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5); break
    except Exception:
        pass
else:
    print("".join(open(f"{WORK}/server.log").readlines()[-40:])); raise SystemExit("❌ The server didn't start: send the lines above to Claude.")
req = urllib.request.Request("http://localhost:8000/v1/chat", method="POST",
    data=json.dumps({"messages": [{"role": "user", "content": "Who are you?"}], "max_tokens": 60}).encode(),
    headers={"content-type": "application/json", "authorization": f"Bearer {KEY}"})
try:
    print("🤖", json.loads(urllib.request.urlopen(req, timeout=600).read())["reply"])
    # Which voice Virgo speaks with (and why, if it isn't Virgo-1.0-Angkor-Voice)
    for line in open("../server.log", errors="replace"):
        if any(k in line for k in ("🗣️ Voice", "Virgo-1.0-Angkor-Voice", "Virgo voice", "VoxCPM", "Running on device")):
            print(line.rstrip()[:300])
except Exception as err:
    print("".join(open(f"{WORK}/server.log").readlines()[-40:])); raise SystemExit(f"❌ {err}: send the lines above to Claude.")
CF = f"{WORK}/cloudflared"
if not os.path.exists(CF):
    subprocess.run(f"wget -q -O {CF} https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x {CF}", shell=True)
subprocess.run("pkill -f 'cloudflared tunnel'", shell=True)
log = f"{WORK}/tunnel.log"
if TUNNEL:
    # The token goes in the environment, never on the command line or in the output.
    subprocess.Popen(f"{CF} tunnel --no-autoupdate run > {log} 2>&1", shell=True, env={**os.environ, "TUNNEL_TOKEN": TUNNEL})
    for _ in range(60):
        time.sleep(2)
        if "Registered tunnel connection" in open(log).read(): break
    else:
        print(open(log).read()[-2000:]); raise SystemExit("❌ Your tunnel didn't connect: check the CF_TUNNEL_TOKEN secret (lines above).")
    url = VIRGO_HOST.rstrip("/")
else:
    subprocess.Popen(f"{CF} tunnel --url http://localhost:8000 > {log} 2>&1", shell=True)
    url = None
    for _ in range(60):
        time.sleep(2)
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(log).read())
        if m: url = m.group(0); break
    if not url:
        print(open(log).read()[-2000:]); raise SystemExit("❌ No public address: send the lines above to Claude.")
print("\n✅ Virgo server is live (chat + real-time voice) at", url)
if TUNNEL and SAVED_KEY:
    print("Your own address and saved key: nothing to change in Cloudflare (VIRGO_API_URL =", url + ", VIRGO_API_KEY = your saved key).")
else:
    print("Add these as Cloudflare Worker SECRETS:")
    print("VIRGO_API_URL =", url)
    print("VIRGO_API_KEY =", "your saved VIRGO_API_KEY secret" if SAVED_KEY else KEY)

In [ ]:
# 3️⃣ Keep the session awake (stop this cell, or the session, when you're done)
import time, urllib.request
while True:
    time.sleep(300)
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=10)
        print(time.strftime("%H:%M"), "✅ Virgo server running")
    except Exception as err:
        print(time.strftime("%H:%M"), "⚠️ server not answering:", err, "— see", f"{WORK}/server.log")